<a href="https://colab.research.google.com/github/panashemuringa/MURINGA-PANASHE-P-HASTS-211-ASSIGNMENT/blob/main/Muringa_Panashe_Assignment_(Financial_Econometrics).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Muringa Panashe Priscilla R2420866
## HASTS211 Assignment
## Financial Econometrics
### Best-Practices Handbook: Challenges in Time Series Modeling

This notebook covers four challenges that come up when modeling financial time series:
1. Multicollinearity
2. Skewness
3. Sensitivity to Outliers
4. Overfitting

We use daily stock return data for **AAPL** (Apple, primary), **MSFT** (Microsoft), **NVDA** (NVIDIA), and **AMD** (Advanced Micro Devices) — four major technology sector stocks — downloaded from Yahoo Finance.

Technology stocks are driven by common macro forces: semiconductor cycles, AI adoption waves, interest-rate sensitivity of high-growth valuations, and platform concentration risk. These forces create strong structural co-movement that makes the sector a compelling test case for all four modelling challenges.

In [ ]:
!pip install yfinance statsmodels scikit-learn numpy pandas matplotlib seaborn scipy --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats
from sklearn.linear_model import LinearRegression, Ridge, ElasticNet
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

print('libraries loaded')

In [ ]:
tickers = ['AAPL', 'MSFT', 'NVDA', 'AMD']
prices  = yf.download(tickers, start='2018-01-01', end='2025-12-31', auto_adjust=True)['Close']
returns = np.log(prices / prices.shift(1)).dropna()
returns.columns = tickers
print('shape:', returns.shape)
returns.head()

---
## Challenge 1: Multicollinearity

### Definition

Multicollinearity occurs when predictor variables in a regression are highly correlated, making it difficult for OLS to isolate the individual effect of each predictor. The standard diagnostic is the **Variance Inflation Factor (VIF)**:

$$\text{VIF}_j = \frac{1}{1 - R_j^2}$$

where $R_j^2$ is obtained by regressing the $j$-th predictor on all other predictors. VIF > 10 indicates severe multicollinearity.

A complementary approach is **Principal Component Analysis (PCA)**. The proportion of variance explained by each component reveals how much redundant information the predictors share: if the first principal component explains a large share of the variance, the predictors are nearly collinear.

### Description

Tech mega-caps are tightly linked through common macro exposures: Federal Reserve rate decisions directly compress or expand the present value of growth earnings, AI infrastructure spend benefits MSFT, NVDA, and AMD in tandem, and risk-off episodes cause simultaneous selloffs across all four. When all four are included as predictors in a regression of AAPL returns, the model cannot distinguish between MSFT's cloud earnings surprise and NVDA's data-centre-driven rally — they all move for the same macro reason at the same time. The result is unstable coefficient estimates that flip sign with small changes to the sample window.

In [ ]:
X = returns[['MSFT', 'NVDA', 'AMD']]
y = returns['AAPL']

X_const = sm.add_constant(X)
model   = sm.OLS(y, X_const).fit()
print(model.summary())

In [ ]:
vif_df = pd.DataFrame()
vif_df['variable'] = X.columns
vif_df['VIF']      = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print('VIF Results:')
print(vif_df)
print('')

# PCA on all four tech stocks
scaler_pca   = StandardScaler()
X_all_scaled = scaler_pca.fit_transform(returns)
pca = PCA()
pca.fit(X_all_scaled)

explained = pca.explained_variance_ratio_
print('PCA Explained Variance Ratio:')
for i, ev in enumerate(explained):
    print(f'  PC{i+1}: {ev:.4f} ({ev*100:.1f}%)')
print(f'\nFirst PC alone explains {explained[0]*100:.1f}% of variance — confirms a single dominant factor (tech/macro sentiment)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(returns.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=axes[0])
axes[0].set_title('Correlation Matrix - Tech Stock Returns')

cumulative = np.cumsum(explained)
axes[1].bar(range(1, 5), explained * 100, color='steelblue', alpha=0.7, label='Individual')
axes[1].plot(range(1, 5), cumulative * 100, 'r-o', label='Cumulative')
axes[1].set_xlabel('Principal Component')
axes[1].set_ylabel('Variance Explained (%)')
axes[1].set_title('PCA Scree Plot - Tech Stocks')
axes[1].legend()

plt.tight_layout()
plt.savefig('multicollinearity_tech.png', dpi=120)
plt.show()

### Diagnosis

- Compute VIF for each predictor — values above 10 signal severe collinearity.
- Run a **PCA scree plot** on all predictors. If the first principal component captures most of the variance, the predictors are nearly collinear and essentially measuring the same thing — in this case, broad tech/AI sentiment. This is a dataset-level view of multicollinearity that complements the per-variable VIF.
- Inspect pairwise correlations — correlations above 0.8 between predictors warrant investigation.

### Damage

When MSFT, NVDA, and AMD are used simultaneously as predictors of AAPL returns, the regression cannot isolate which company-specific factor is driving AAPL. Hedge ratios computed from such a model will be numerically unstable: a small revision to NVDA's recent return history will cause the estimated AAPL-NVDA coefficient to swing wildly while the AAPL-MSFT coefficient absorbs an equal and opposite swing. For a tech-sector equity portfolio manager trying to hedge individual name exposure, these unstable hedge ratios are operationally unmanageable and can result in unintended residual risk.

### Directions

1. **PCA regression** — replace the correlated predictors with their principal components. PC1 captures the common tech-sentiment factor; PC2 and beyond capture residual company-specific variation (e.g., NVDA's semiconductor cycle exposure vs. MSFT's cloud subscription revenues). This eliminates collinearity by construction.
2. **ElasticNet regularisation** — combines Ridge and Lasso penalties, handling groups of correlated variables more gracefully than either alone.
3. **Include a common factor directly** — if the underlying driver is the NASDAQ or an AI/semiconductor index, model it explicitly and remove the redundant stock predictors.

In [ ]:
scaler    = StandardScaler()
X_scaled  = scaler.fit_transform(X)

ols   = LinearRegression().fit(X_scaled, y)
ridge = Ridge(alpha=1.0).fit(X_scaled, y)
enet  = ElasticNet(alpha=0.01, l1_ratio=0.5).fit(X_scaled, y)

coef_compare = pd.DataFrame({
    'variable':        tickers[1:],
    'OLS coef':        ols.coef_.round(5),
    'Ridge coef':      ridge.coef_.round(5),
    'ElasticNet coef': enet.coef_.round(5)
})
print(coef_compare)
print('')
print('ElasticNet combines L1 and L2 penalties — useful when predictors are grouped and correlated')

---
## Challenge 2: Skewness

### Definition

Skewness quantifies asymmetry in a return distribution:

$$\text{Skewness} = \frac{\frac{1}{n}\sum_{i=1}^{n}(x_i - \bar{x})^3}{\left(\frac{1}{n}\sum_{i=1}^{n}(x_i - \bar{x})^2\right)^{3/2}}$$

Negative skewness indicates that the left tail (losses) is heavier than the right tail (gains). In technology markets, this manifests as sharp risk-off episodes — Fed rate hike surprises, earnings misses on inflated multiples, or AI sentiment reversals — producing sudden equity losses that are not mirrored by gradual recoveries of equal magnitude.

We test for departure from normality using the **Kolmogorov-Smirnov (KS) test**, which compares the empirical cumulative distribution of returns to a fitted normal distribution. The KS statistic is the maximum gap between the two CDFs. Unlike the Jarque-Bera test, the KS test is sensitive to departures across the full distribution — not just the tails — making it a broader diagnostic.

### Description

Technology stocks, particularly high-multiple growth names like NVDA and AMD, are acutely sensitive to interest rate changes: when the Fed signals tighter-than-expected policy, discounted cash flow models produce large valuation reductions instantly, while recoveries require both policy reversals and fundamental earnings delivery over subsequent quarters. The 2022 rate-hike cycle cut NVDA's stock by over 60% in under 12 months. This asymmetry between speed of decline and pace of recovery permanently imprints negative skewness on tech return distributions.

In [ ]:
print('Skewness, Kurtosis, and Kolmogorov-Smirnov Normality Tests')
print('='*65)

for ticker in tickers:
    r  = returns[ticker]
    sk = r.skew()
    ku = r.kurtosis()

    # KS test against a normal distribution fitted to the data
    ks_stat, ks_p = stats.kstest(r, 'norm', args=(r.mean(), r.std()))
    result = 'reject normality' if ks_p < 0.05 else 'cannot reject normality'

    print(f'{ticker}:  skewness = {sk:.4f},  excess kurtosis = {ku:.4f}')
    print(f'       KS stat = {ks_stat:.4f},  p = {ks_p:.4e}  => {result}')
    print('')

print('All tech stocks reject normality — asymmetric fat-tailed distributions dominate')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

r = returns['AAPL']

# Histogram
axes[0].hist(r, bins=60, density=True, color='steelblue', alpha=0.6, label='AAPL returns')
x = np.linspace(r.min(), r.max(), 200)
axes[0].plot(x, stats.norm.pdf(x, r.mean(), r.std()), 'r-', lw=2, label='Normal fit')
axes[0].set_title('AAPL Log Returns vs Normal Distribution')
axes[0].set_xlabel('Log Return')
axes[0].set_ylabel('Density')
axes[0].legend()

# Empirical CDF vs Normal CDF (visualises the KS test)
sorted_r   = np.sort(r)
ecdf       = np.arange(1, len(sorted_r) + 1) / len(sorted_r)
normal_cdf = stats.norm.cdf(sorted_r, r.mean(), r.std())

axes[1].plot(sorted_r, ecdf,       color='steelblue', lw=2, label='Empirical CDF')
axes[1].plot(sorted_r, normal_cdf, color='red', lw=2, linestyle='--', label='Normal CDF')
axes[1].set_title('Empirical vs Normal CDF - AAPL (KS Test Visualised)')
axes[1].set_xlabel('Log Return')
axes[1].set_ylabel('Cumulative Probability')
axes[1].legend()

plt.tight_layout()
plt.savefig('skewness_tech.png', dpi=120)
plt.show()
print('The gap between empirical and normal CDF in the left tail confirms negative skewness')

### Diagnosis

- Compute sample skewness — values below -0.5 indicate meaningful left asymmetry.
- Apply the **Kolmogorov-Smirnov test**, which tests whether the empirical distribution matches a fitted normal. Visualising the empirical vs. normal CDF makes the departure immediately visible.
- A QQ plot is a complementary visual — points deviating below the diagonal at the left end confirm negative skewness.

### Damage

For a tech-sector options desk or a systematic equity fund, underestimating left-tail skewness means underpricing downside puts on tech stocks and mis-sizing stop-loss orders. A risk model that assumes AAPL returns are normally distributed will estimate a 1% daily VaR that is materially too low during rate-driven drawdowns. When the 2022 repricing hit AAPL (which shed roughly 27% in 2022), every portfolio that sized positions on normal-distribution VaR found itself significantly over-exposed. Negative skewness in tech is not a rare anomaly — it is the structural consequence of growth stock valuation mechanics.

### Directions

1. **Fit a Student-t distribution** — its heavier tails better capture the fat-tailed, negatively skewed behaviour of tech returns.
2. **Use non-parametric risk measures** — historical simulation VaR and Expected Shortfall make no distributional assumptions and directly account for observed skewness.
3. **GARCH with asymmetric variance** — models like EGARCH or GJR-GARCH allow negative return shocks to increase volatility more than positive shocks of the same size, capturing the leverage effect common in high-multiple tech stocks.

---
## Challenge 3: Sensitivity to Outliers

### Definition

A model is sensitive to outliers when a small number of extreme observations disproportionately influence the estimated parameters. In OLS, this is measured with **Cook's Distance**:

$$D_i = \frac{\sum_{j=1}^{n}(\hat{y}_j - \hat{y}_{j(-i)})^2}{p \cdot MSE}$$

Values above $4/n$ flag influential observations.

An alternative is the **leverage** ($h_{ii}$), the diagonal of the hat matrix $\mathbf{H} = \mathbf{X}(\mathbf{X}^\top \mathbf{X})^{-1}\mathbf{X}^\top$, which measures how extreme an observation's predictor values are relative to the predictor space centroid. High leverage observations have high potential to distort the regression even if their residual appears modest:

$$\text{High leverage: } h_{ii} > \frac{2p}{n}$$

### Description

Technology stocks experience extreme price moves during macro shocks and company-specific events. The COVID-19 sell-off in February–March 2020, the Fed's January 2022 pivot signal, and NVDA's earnings-driven gap-ups following the 2023 AI boom all created multi-standard-deviation days that are qualitatively different from normal trading. OLS does not distinguish between these exceptional days and routine trading sessions, so these extreme returns exert a strong pull on the estimated regression line.

In [ ]:
X_simple = sm.add_constant(returns[['MSFT']])
y_simple  = returns['AAPL']

model_ols = sm.OLS(y_simple, X_simple).fit()
influence  = model_ols.get_influence()

cooks_d   = influence.cooks_distance[0]
leverage  = influence.hat_matrix_diag
std_resid = influence.resid_studentized_internal

n, p = len(y_simple), 2
cooks_threshold    = 4 / n
leverage_threshold = 2 * p / n

n_outliers      = (np.abs(std_resid) > 3).sum()
n_influential   = (cooks_d > cooks_threshold).sum()
n_high_leverage = (leverage > leverage_threshold).sum()

print(f'Total observations:                    {n}')
print(f'Outliers (|std residual| > 3):         {n_outliers}')
print(f"Influential (Cook's D > 4/n):          {n_influential}")
print(f'High leverage (h_ii > 2p/n):           {n_high_leverage}')
print('')

top5 = np.argsort(cooks_d)[-5:][::-1]
print('Top 5 most influential trading days:')
for i in top5:
    print(f"  {returns.index[i].date()}  AAPL={returns['AAPL'].iloc[i]:.4f}  MSFT={returns['MSFT'].iloc[i]:.4f}  Cook's D={cooks_d[i]:.6f}  leverage={leverage[i]:.4f}")

In [ ]:
influential_mask = cooks_d > cooks_threshold

X_clean     = sm.add_constant(returns['MSFT'][~influential_mask])
model_clean = sm.OLS(returns['AAPL'][~influential_mask], X_clean).fit()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

x_range = np.linspace(returns['MSFT'].min(), returns['MSFT'].max(), 200)
axes[0].scatter(returns['MSFT'][~influential_mask], returns['AAPL'][~influential_mask],
                alpha=0.2, s=5, color='steelblue', label='normal obs')
axes[0].scatter(returns['MSFT'][influential_mask], returns['AAPL'][influential_mask],
                alpha=0.9, s=25, color='red', label='influential obs')
axes[0].plot(x_range, model_ols.params[0]   + model_ols.params[1]   * x_range, 'r-',  lw=2, label='OLS full data')
axes[0].plot(x_range, model_clean.params[0] + model_clean.params[1] * x_range, 'g--', lw=2, label='OLS clean data')
axes[0].set_title('Effect of Outliers on AAPL-MSFT Regression')
axes[0].set_xlabel('MSFT return')
axes[0].set_ylabel('AAPL return')
axes[0].legend(fontsize=8)

axes[1].bar(range(len(cooks_d)), cooks_d, color='steelblue', alpha=0.6)
axes[1].axhline(cooks_threshold, color='red', linestyle='--', label='threshold = 4/n')
axes[1].set_title("Cook's Distance")
axes[1].set_xlabel('Observation')
axes[1].legend()

# Influence plot: leverage vs std residuals (bubble = Cook's D)
axes[2].scatter(leverage, std_resid, s=cooks_d * 5000, alpha=0.3, color='darkorange')
axes[2].axhline(3,  color='red', linestyle='--', lw=1)
axes[2].axhline(-3, color='red', linestyle='--', lw=1, label='+-3 std residual')
axes[2].axvline(leverage_threshold, color='green', linestyle='--', lw=1, label='leverage threshold')
axes[2].set_title("Influence Plot (bubble size = Cook's D)")
axes[2].set_xlabel('Leverage (h_ii)')
axes[2].set_ylabel('Standardised Residual')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig('outlier_tech.png', dpi=120)
plt.show()

### Diagnosis

- Check standardised residuals — values beyond +/-3 signal outliers.
- Compute Cook's Distance — values above $4/n$ identify observations that strongly influence the regression coefficients.
- Compute **leverage** ($h_{ii}$) — values above $2p/n$ identify observations that are extreme in the predictor space.
- An **influence plot** — leverage on the x-axis, standardised residuals on the y-axis, bubble size proportional to Cook's D — combines all three diagnostics in a single visualisation.

### Damage

For tech equity portfolio managers and systematic traders, outlier sensitivity means that beta estimates computed from full-sample models are unreliable for normal market conditions. An AAPL-MSFT beta calibrated over a period that includes the March 2020 COVID crash will reflect crisis co-movement dynamics — artificially inflating the beta because both stocks fell simultaneously for macro rather than fundamental reasons. Using this inflated beta to size hedges in a calm market leads to over-hedging, tying up capital and dragging on returns unnecessarily.

### Directions

1. **Winsorisation** — cap extreme returns at the 1st and 99th percentiles before fitting, directly limiting the influence of any single session.
2. **Huber robust regression** — minimises a hybrid loss function that is quadratic for small residuals and linear for large ones, downweighting outliers automatically.
3. **Regime-aware modelling** — identify macro shock periods and fit separate models for crisis and non-crisis regimes.

---
## Challenge 4: Overfitting

### Definition

Overfitting occurs when a model learns the noise in the training data rather than the true signal, resulting in good in-sample performance but poor out-of-sample forecasting.

The **bias-variance tradeoff** frames this formally:

$$\text{Expected MSE} = \text{Bias}^2 + \text{Variance} + \sigma^2_{\varepsilon}$$

Complex models minimise bias but inflate variance. The optimal model minimises the sum.

**ElasticNet regularisation** controls this tradeoff with two penalties:

$$\text{ElasticNet Loss} = \text{RSS} + \alpha \left[ (1 - \rho) \sum_j \beta_j^2 + \rho \sum_j |\beta_j| \right]$$

where $\alpha$ controls overall regularisation strength and $\rho$ (l1_ratio) interpolates between Ridge ($\rho = 0$) and Lasso ($\rho = 1$). ElasticNet is particularly useful when predictors are grouped and correlated, as with tech stocks.

### Description

Technology markets are subject to sharp regime shifts driven by macro policy and thematic narratives: the zero-rate era (2018–2021) rewarded high-multiple growth equities; the 2022 rate-hike regime punished them; the 2023 AI boom selectively re-rated semiconductor and cloud names. A model trained on one regime will overfit its idiosyncratic dynamics and fail in the next. NVDA's 2023 earnings trajectory — driven by data-centre AI demand that had no precedent in the training data — was structurally impossible to forecast from pre-2023 patterns.

In [ ]:
X_data = returns['MSFT'].values.reshape(-1, 1)
y_data = returns['AAPL'].values

split   = int(len(X_data) * 0.7)
X_train, X_test = X_data[:split], X_data[split:]
y_train, y_test = y_data[:split], y_data[split:]

print(f'Training observations: {len(X_train)}')
print(f'Test observations:     {len(X_test)}')
print('')
print(f'{"Degree":<10}{"Train RMSE":<16}{"Test RMSE":<16}')
print('-'*42)

for degree in [1, 2, 4, 8]:
    poly   = PolynomialFeatures(degree=degree)
    scaler = StandardScaler()

    Xtr_poly = scaler.fit_transform(poly.fit_transform(X_train))
    Xte_poly = scaler.transform(poly.transform(X_test))

    reg = LinearRegression().fit(Xtr_poly, y_train)

    train_rmse = np.sqrt(mean_squared_error(y_train, reg.predict(Xtr_poly)))
    test_rmse  = np.sqrt(mean_squared_error(y_test,  reg.predict(Xte_poly)))

    print(f'{degree:<10}{train_rmse:<16.6f}{test_rmse:<16.6f}')

print('')
print('Train RMSE falls consistently; test RMSE deteriorates at higher degrees - overfitting.')

In [ ]:
degrees     = [1, 2, 3, 4, 5, 6, 7, 8]
train_errors = []
test_errors  = []

for degree in degrees:
    poly   = PolynomialFeatures(degree=degree)
    scaler = StandardScaler()
    Xtr    = scaler.fit_transform(poly.fit_transform(X_train))
    Xte    = scaler.transform(poly.transform(X_test))
    reg    = LinearRegression().fit(Xtr, y_train)
    train_errors.append(np.sqrt(mean_squared_error(y_train, reg.predict(Xtr))))
    test_errors.append(np.sqrt(mean_squared_error(y_test,   reg.predict(Xte))))

plt.figure(figsize=(9, 5))
plt.plot(degrees, train_errors, 'g-o', label='Train RMSE')
plt.plot(degrees, test_errors,  'r-s', label='Test RMSE')
plt.xlabel('Polynomial Degree (Model Complexity)')
plt.ylabel('RMSE')
plt.title('Overfitting: Train vs Test RMSE (AAPL on MSFT)')
plt.legend()
plt.tight_layout()
plt.savefig('overfitting_tech.png', dpi=120)
plt.show()

### Diagnosis

- Hold out a test set that was not used in training. A substantial gap between train and test RMSE is the primary indicator of overfitting.
- Compare **adjusted $R^2$** across models of differing complexity — if it falls when a variable is added, the variable is not providing genuine predictive power.
- Use **AIC and BIC** for nested model comparison — both penalise additional parameters, with BIC applying a stronger penalty for large samples.
- In time series, use **walk-forward (expanding window) cross-validation** rather than random k-fold, to avoid training on future data.

### Damage

For technology sector modelling, overfitting is particularly dangerous because the sector is narrative-driven. A model trained on 2018–2021 AAPL-MSFT co-movement learned a specific low-rate, cloud-expansion regime. When rates rose sharply in 2022, the relationship between the two stocks shifted because their valuation sensitivities differ: MSFT's stable subscription revenues provided partial insulation while AAPL faced consumer spending headwinds. Models that memorised the calm co-movement of 2019–2021 systematically mispriced this divergence, producing inaccurate pair-trade signals and risk estimates throughout 2022.

### Directions

1. **ElasticNet regularisation** — particularly suitable for tech returns because it handles groups of correlated predictors better than Ridge or Lasso alone, and reduces overfitting by penalising large coefficients.
2. **Walk-forward cross-validation** — ensures that out-of-sample evaluation mirrors the live environment by never training on data beyond the forecast origin.
3. **Prefer parsimonious models** — GARCH(1,1) with a simple mean equation consistently outperforms more elaborate models in out-of-sample volatility forecasting for tech equities.

In [ ]:
poly   = PolynomialFeatures(degree=8)
scaler = StandardScaler()
Xtr    = scaler.fit_transform(poly.fit_transform(X_train))
Xte    = scaler.transform(poly.transform(X_test))

ols_overfit = LinearRegression().fit(Xtr, y_train)
ridge_model = Ridge(alpha=1.0).fit(Xtr, y_train)
enet_model  = ElasticNet(alpha=0.001, l1_ratio=0.5).fit(Xtr, y_train)

print('Degree-8 model: OLS vs Ridge vs ElasticNet')
for name, model in [('OLS', ols_overfit), ('Ridge', ridge_model), ('ElasticNet', enet_model)]:
    tr = np.sqrt(mean_squared_error(y_train, model.predict(Xtr)))
    te = np.sqrt(mean_squared_error(y_test,  model.predict(Xte)))
    n_nonzero = np.sum(np.abs(model.coef_) > 1e-10)
    print(f'{name:<12} - train RMSE: {tr:.6f},  test RMSE: {te:.6f},  non-zero coefs: {n_nonzero}')

print('')
print('ElasticNet combines Ridge stability with Lasso sparsity - reduces test RMSE while zeroing redundant terms')

---
## Summary

| Challenge | How to detect | Suggested fix |
|-----------|--------------|---------------|
| Multicollinearity | VIF > 10, PCA scree plot dominated by first component | ElasticNet, PCA regression, include NASDAQ factor directly |
| Skewness | KS test, sample skewness < -0.5, empirical vs normal CDF gap | Student-t distribution, historical simulation, EGARCH |
| Sensitivity to Outliers | Cook's Distance > 4/n, high leverage h_ii, influence plot | Winsorisation, Huber regression, regime-aware models |
| Overfitting | Train vs test RMSE gap, adjusted R-squared | ElasticNet regularisation, walk-forward CV, parsimonious models |

Technology stocks present a particularly concentrated set of modelling challenges: common macro and AI-sentiment exposure creates severe multicollinearity, rate-driven valuation compressions create extreme skewness, and sharp regime changes between growth and value environments cause overfitting. Careful application of these diagnostic tools is essential before any tech-sector model is deployed in practice.

---
## Non-Technical Report

### Section 1 — What I Found

This analysis examined the daily stock returns of four major technology companies — Apple (AAPL), Microsoft (MSFT), NVIDIA (NVDA), and AMD — and identified four key modelling problems that can lead to flawed risk management and investment decisions if left unaddressed.

- **All four stocks are driven by the same macro factor:** AAPL, MSFT, NVDA, and AMD all respond dramatically to Federal Reserve interest rate signals, AI infrastructure investment cycles, and broader tech-sector sentiment. Principal Component Analysis confirmed that a single underlying factor explains the majority of their joint return variation. A model treating them as four independent predictors is building on a false premise.
- **Large losses happen far more suddenly than large gains of the same size:** All four tech stocks showed negative skewness. Rate-driven valuation compressions are sudden and steep — NVDA fell over 60% in 2022 — while recoveries require both policy reversals and earnings delivery over subsequent quarters. Standard models that assume symmetric returns will routinely underestimate how bad a bad year can be.
- **A small number of extraordinary days dominate every risk estimate:** The COVID market crash in February–March 2020, the Fed's January 2022 rate pivot, and NVDA's AI-driven earnings gap-ups in 2023 created extreme return observations that pull regression models heavily toward those specific conditions, distorting estimates for ordinary trading periods.
- **Models trained on one tech regime fail in the next:** The 2019–2021 zero-rate growth regime and the 2022 rate-hike value rotation are fundamentally different environments. A model fitted on the first will fail in the second — not because it is poorly constructed, but because it memorised the wrong regime.

### Section 2 — Recommended Course of Action

- **Do not treat the four tech stocks as independent risk factors:** They are all proxies for broad tech and macro sentiment. Portfolio construction and risk budgeting should reflect this — holding all four does not provide meaningful diversification within the technology sector.
- **Set loss provisions and hedge budgets based on actual tail risk:** Because tech returns are negatively skewed, Value-at-Risk based on the normal distribution will understate true downside. Historical simulation, which reads off actual past losses, is more appropriate for tech equity risk management.
- **Separate macro shock periods from normal periods in model calibration:** The COVID crash and the 2022 rate-hike cycle are structural events that should be modelled separately, not allowed to distort a single model meant to describe everyday tech price dynamics.
- **Use simple, validated models over complex ones:** A parsimonious GARCH model evaluated out-of-sample consistently outperforms complex models calibrated on historical data. Complexity that does not survive out-of-sample testing should not be deployed.

### Section 3 — What Drove the Problems

- **Single factor dominance:** All four companies derive a large share of their value from future growth expectations that are discounted by the same interest rate. When the Fed raises rates, every growth stock's present value falls simultaneously, creating near-perfect correlation.
- **Asymmetric losses:** Valuation compressions driven by rising rates happen in weeks as investors reprice models. Recoveries require not just rate cuts but also several quarters of earnings delivery to rebuild confidence. This structural asymmetry permanently imprints negative skewness on tech equity returns.
- **Crisis-driven outliers:** The 2020 COVID crash was the fastest equity drawdown in modern history. NVDA's 2023 AI-driven earnings surge was unprecedented in scale for a company of its size. These were extraordinary events with no direct historical analogue in the prior data.
- **Regime instability:** Technology stocks went from the beneficiaries of a decade of zero rates (2012–2021) to the primary victims of the fastest rate-hiking cycle in 40 years (2022), then partially recovered on AI optimism (2023). Each transition represented a structural break that historical models could not anticipate.

In summary, technology stocks behave more like a leveraged bet on macro policy and AI-theme adoption than a diversified equity sector, with fat-tailed, negatively skewed returns and high sensitivity to interest rate regimes and sentiment shifts.

---
## References

- Belsley, D.A., Kuh, E., & Welsch, R.E. (1980). *Regression Diagnostics*. John Wiley & Sons.
- Huber, P.J. (1973). Robust Regression: Asymptotics, Conjectures and Monte Carlo. *The Annals of Statistics*, 1(5), 799–821.
- Hull, J.C. (2018). *Options, Futures, and Other Derivatives* (10th ed.). Pearson.
- James, G., Witten, D., Hastie, T., & Tibshirani, R. (2021). *An Introduction to Statistical Learning* (2nd ed.). Springer.
- Jolliffe, I.T. (2002). *Principal Component Analysis* (2nd ed.). Springer.
- Massey, F.J. (1951). The Kolmogorov-Smirnov Test for Goodness of Fit. *Journal of the American Statistical Association*, 46(253), 68–78.
- Zou, H., & Hastie, T. (2005). Regularization and Variable Selection via the Elastic Net. *Journal of the Royal Statistical Society: Series B*, 67(2), 301–320.